# ABC Company — Module 1 End Project
Comprehensive employee analysis with preprocessing, EDA, and visualizations.

**Author:** <Your Name>

**Dataset:** Provided by ABC Company (458 rows × 9 columns)

**Overview:**
1. Preprocessing (fix `height` column)
2. Team distribution + % split
3. Position-wise segregation
4. Predominant age group
5. Highest salary expenditure (team & position)
6. Correlation: age vs salary (with visualization)
7. Data story & insights


## 0) Imports & Utilities

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

def find_dataset_file():
    # Look for CSV/XLSX in current working directory and /mnt/data
    candidates = []
    for base in ['.', '/mnt/data']:
        if os.path.isdir(base):
            for f in os.listdir(base):
                if f.lower().endswith(('.csv', '.xlsx')):
                    candidates.append(os.path.join(base, f))
    return candidates

def load_dataset(path: str):
    if path.lower().endswith('.csv'):
        df = pd.read_csv(path)
    else:
        df = pd.read_excel(path)
    return df

def preview(df: pd.DataFrame, n=5):
    display(df.head(n))
    display(df.info())
    display(df.describe(include='all'))


## 1) Load Dataset

In [ ]:
candidates = find_dataset_file()
if len(candidates) == 0:
    print('No dataset found. Please upload a .csv or .xlsx file to this chat.\n' 
          'Tip: name it abc_employees.csv or abc_employees.xlsx')
else:
    # Pick the first candidate by default (edit if needed)
    dataset_path = candidates[0]
    print('Using dataset:', dataset_path)
    df = load_dataset(dataset_path)
    print('Shape:', df.shape)
    preview(df)


## 2) Preprocessing — Fix `height` column
Replace the `height` column values with random integers in [150, 180].

In [ ]:
def fix_height_column(df: pd.DataFrame, low=150, high=180, column='height', seed=None):
    if seed is not None:
        np.random.seed(seed)
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found in dataset. Available: {list(df.columns)}")
    # Generate random integers inclusive of [low, high]
    df[column] = np.random.randint(low, high + 1, size=len(df))
    return df

# Execute only if df exists
try:
    df
    df = fix_height_column(df)
    print('`height` column replaced with random integers [150, 180].')
    preview(df)
    # Save a cleaned copy
    cleaned_path = '/mnt/data/abc_employees_clean.csv'
    df.to_csv(cleaned_path, index=False)
    print('Cleaned dataset saved to:', cleaned_path)
except NameError:
    print('Dataset not loaded yet. Upload the file, then re-run the cells.')


## 3) Analysis — Team distribution & percentage split

In [ ]:
try:
    team_counts = df['team'].value_counts(dropna=False).sort_values(ascending=False)
    team_pct = (team_counts / len(df) * 100).round(2)
    display(pd.DataFrame({'count': team_counts, 'percent': team_pct}))
except Exception as e:
    print('Run after dataset is loaded & columns verified. Error:', e)


## 4) Analysis — Position-wise segregation

In [ ]:
try:
    position_counts = df['position'].value_counts(dropna=False).sort_values(ascending=False)
    display(position_counts.to_frame('count'))
except Exception as e:
    print('Run after dataset is loaded & columns verified. Error:', e)


## 5) Analysis — Predominant age group

In [ ]:
try:
    # Define age bins (edit if needed)
    bins = [0, 20, 25, 30, 35, 40, 45, 50, 60, 120]
    labels = ['<20','20-25','26-30','31-35','36-40','41-45','46-50','51-60','60+']
    df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=True, include_lowest=True)
    age_group_counts = df['age_group'].value_counts().sort_index()
    display(age_group_counts.to_frame('count'))
    top_group = age_group_counts.idxmax()
    print('Predominant age group:', top_group)
except Exception as e:
    print('Run after dataset is loaded & columns verified. Error:', e)


## 6) Analysis — Highest salary expenditure (team & position)

In [ ]:
try:
    # Team with highest total salary
    team_salary = df.groupby('team')['salary'].sum().sort_values(ascending=False)
    print('Team salary totals:')
    display(team_salary.to_frame('total_salary'))
    print('\nTop team by salary spend:', team_salary.index[0], '=>', team_salary.iloc[0])

    # Position with highest total salary
    pos_salary = df.groupby('position')['salary'].sum().sort_values(ascending=False)
    print('\nPosition salary totals:')
    display(pos_salary.to_frame('total_salary'))
    print('\nTop position by salary spend:', pos_salary.index[0], '=>', pos_salary.iloc[0])
except Exception as e:
    print('Run after dataset is loaded & columns verified. Error:', e)


## 7) Correlation — Age vs Salary (with visualization)

In [ ]:
try:
    corr_val = df[['age','salary']].corr().loc['age','salary']
    print('Correlation (age vs salary):', round(corr_val, 4))
    # Scatter plot
    plt.figure()
    plt.scatter(df['age'], df['salary'])
    plt.title('Age vs Salary')
    plt.xlabel('Age')
    plt.ylabel('Salary')
    plt.tight_layout()
    plt.savefig('/mnt/data/plot_age_vs_salary.png', dpi=150)
    plt.show()
    print('Saved figure to /mnt/data/plot_age_vs_salary.png')
except Exception as e:
    print('Run after dataset is loaded & columns verified. Error:', e)


## 8) Visualizations — Team distribution & position-wise, age groups

In [ ]:
try:
    # Team distribution bar chart
    plt.figure()
    team_counts.plot(kind='bar')
    plt.title('Employees per Team')
    plt.xlabel('Team')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig('/mnt/data/plot_team_distribution.png', dpi=150)
    plt.show()
    print('Saved figure to /mnt/data/plot_team_distribution.png')

    # Position distribution bar chart
    plt.figure()
    position_counts.plot(kind='bar')
    plt.title('Employees by Position')
    plt.xlabel('Position')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig('/mnt/data/plot_position_distribution.png', dpi=150)
    plt.show()
    print('Saved figure to /mnt/data/plot_position_distribution.png')

    # Age group distribution bar chart
    plt.figure()
    df['age_group'].value_counts().sort_index().plot(kind='bar')
    plt.title('Employees by Age Group')
    plt.xlabel('Age Group')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig('/mnt/data/plot_age_groups.png', dpi=150)
    plt.show()
    print('Saved figure to /mnt/data/plot_age_groups.png')
except Exception as e:
    print('Run after dataset is loaded & columns verified. Error:', e)


## 9) Data Story — Insights

- Summarize key findings:
  - Which teams and positions dominate headcount?
  - Predominant age group and implications for workforce planning.
  - Highest salary expenditure (teams & positions) and potential cost centers.
  - Correlation between age and salary—interpret direction & magnitude.

- Contextualize any limitations (e.g., synthetic height replacement).


## 10) Save cleaned dataset (already saved) & finish